# LSSTCam — HEALPix Visit Counts: Global Map and Monthly Subplots

**Purpose:**  
Load the visit tables previously downloaded from USDF (Butler and consDb parquet files),
build a canonical merged visit list with pointing coordinates, and produce:

1. **A global all-time HEALPix map** of total visit counts (log scale) over the full survey period,
   using a Mollweide projection with Galactic plane trace and DDF markers.
2. **Monthly subplots** (one subplot per calendar month) showing the evolution of visit counts
   per month in log scale, all-bands combined.
3. **Band-resolved monthly subplots** (2×3 panel: u, g, r / i, z, y) for each month,
   following the style of `2026-03-10_ConsDB_LSSTCam_HEALPix_Monthly_subplots.ipynb`.

**Data sources (local parquet files from USDF):**
- Butler `visitTable` with tract/patch geometry
- consDb `visitTable` with tract/patch geometry

Both parquets must contain pointing coordinates (`s_ra`, `s_dec`) and a `band` column.
Observation date is derived from the `day_obs` integer field (YYYYMMDD format).

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS)  
**Creation date:** 2026-04-09  
**Based on:**
- `05_crosscheck_visitTable_vs_constDb.ipynb` (visit loading & merging logic)
- `2026-03-10_ConsDB_LSSTCam_HEALPix_Monthly_subplots.ipynb` (HEALPix mapping & plotting)

## 0. Imports and global configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import healpy as hp
import warnings

from astropy.coordinates import SkyCoord
import astropy.units as u

warnings.filterwarnings("ignore")

%matplotlib inline

print(f"healpy  version : {hp.__version__}")
print(f"numpy   version : {np.__version__}")
print(f"pandas  version : {pd.__version__}")

## 1. Configuration

In [ ]:
# ── Instrument label ─────────────────────────────────────────────────────────
INSTRUMENT = "LSSTCam"

# ── Data paths ────────────────────────────────────────────────────────────────
DATA_DIR = "data_fromlsst"

# Butler visitTable parquet (with tracts / patches / pointing coordinates)
FILE_BUTLER = os.path.join(DATA_DIR, "visitTable-2025041500138-2026040500856_N58748_WithTracts.parquet")

# consDb visitTable parquet (with tracts / patches / pointing coordinates)
FILE_CONSDB = os.path.join(
    DATA_DIR, "constDbVisitTable-2025041500043-2026040600288_N93006_WithTracts.parquet"
)

print(f"Butler file  : {FILE_BUTLER}")
print(f"  exists     : {os.path.exists(FILE_BUTLER)}")
print(f"consDb file  : {FILE_CONSDB}")
print(f"  exists     : {os.path.exists(FILE_CONSDB)}")

# ── HEALPix resolution ──────────────────────────────────────────────────────
# NSIDE=32  -> pixel size ~1.83 deg  (fast overview)
# NSIDE=64  -> pixel size ~0.92 deg  (good balance)
# NSIDE=128 -> pixel size ~0.46 deg  (finer, slower)
NSIDE = 64
NPIX = hp.nside2npix(NSIDE)
print(f"\nNSIDE={NSIDE}  ->  {NPIX} pixels,  pixel size ~ {np.degrees(hp.nside2resol(NSIDE)):.2f} deg")

# ── LSST bands ───────────────────────────────────────────────────────────────
BANDS = list("ugrizy")

BAND_COLORS = {
    "u": "#9b59b6",  # purple
    "g": "#2ecc71",  # green
    "r": "#e74c3c",  # red
    "i": "#e67e22",  # orange
    "z": "#3498db",  # blue
    "y": "#795548",  # brown
}

BAND_CMAPS = {
    "u": "Purples",
    "g": "Greens",
    "r": "Reds",
    "i": "Oranges",
    "z": "Blues",
    "y": "YlOrBr",
}

# 2x3 grid row layout for band panels
BAND_ROWS = [["u", "g", "r"], ["i", "z", "y"]]

# ── Deep Drilling Fields (ICRS degrees) ─────────────────────────────────────
DDF_NAMES = ["XMM-LSS", "COSMOS", "ECDFS", "ELAIS-S1", "EDFS_a", "EDFS_b"]
DDF_RA_DEG = np.array([35.57, 150.11, 52.98, 9.45, 58.90, 63.60])
DDF_DEC_DEG = np.array([-4.82, 2.23, -28.12, -44.02, -49.32, -47.60])

# ── Matplotlib defaults ──────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.figsize": (12, 6),
        "axes.labelsize": "large",
        "axes.titlesize": "large",
        "xtick.labelsize": "large",
        "ytick.labelsize": "large",
    }
)

print("Configuration done.")

## 2. Load Butler and consDb visit tables

In [ ]:
df_butler = pd.read_parquet(FILE_BUTLER)
df_consdb = pd.read_parquet(FILE_CONSDB)

print("=" * 60)
print(f"Butler visitTable  :  {df_butler.shape[0]:,} rows  x  {df_butler.shape[1]} columns")
print(f"consDb visitTable  :  {df_consdb.shape[0]:,} rows  x  {df_consdb.shape[1]} columns")
print()
print("Butler columns :", df_butler.columns.tolist())
print("consDb columns :", df_consdb.columns.tolist())

### 2.1 Auto-detect key columns

In [ ]:
def detect_col(df, candidates, label):
    """Return the first column name from `candidates` that exists in `df`."""
    for c in candidates:
        if c in df.columns:
            print(f'  [{label}] found column: "{c}"')
            return c
    print(f"  [{label}] WARNING – none of {candidates} found in columns")
    return None


print("Auto-detecting visit column ...")
COL_VISIT_BUTLER = detect_col(df_butler, ["visitId", "visit", "r:visit", "visit_id", "id"], "butler")
COL_VISIT_CONSDB = detect_col(df_consdb, ["visit_id", "visitId", "visit", "r:visit", "id"], "consdb")

print("Auto-detecting RA column ...")
COL_RA_BUTLER = detect_col(df_butler, ["s_ra", "ra", "RA", "boresightRaDec_x", "ra_deg"], "butler")
COL_RA_CONSDB = detect_col(df_consdb, ["s_ra", "ra", "RA", "ra_deg"], "consdb")

print("Auto-detecting Dec column ...")
COL_DEC_BUTLER = detect_col(
    df_butler, ["s_dec", "dec", "Dec", "DEC", "boresightRaDec_y", "dec_deg"], "butler"
)
COL_DEC_CONSDB = detect_col(df_consdb, ["s_dec", "dec", "Dec", "DEC", "dec_deg"], "consdb")

print("Auto-detecting band column ...")
COL_BAND_BUTLER = detect_col(df_butler, ["band", "filter", "physical_filter"], "butler")
COL_BAND_CONSDB = detect_col(df_consdb, ["band", "filter", "physical_filter"], "consdb")

print("Auto-detecting day_obs column ...")
COL_DAYOBS_BUTLER = detect_col(df_butler, ["day_obs", "dayObs", "obs_day"], "butler")
COL_DAYOBS_CONSDB = detect_col(df_consdb, ["day_obs", "dayObs", "obs_day"], "consdb")

print("\nColumn mapping summary:")
print(f"  visit   : butler={COL_VISIT_BUTLER}, consdb={COL_VISIT_CONSDB}")
print(f"  RA      : butler={COL_RA_BUTLER},    consdb={COL_RA_CONSDB}")
print(f"  Dec     : butler={COL_DEC_BUTLER},   consdb={COL_DEC_CONSDB}")
print(f"  band    : butler={COL_BAND_BUTLER},  consdb={COL_BAND_CONSDB}")
print(f"  day_obs : butler={COL_DAYOBS_BUTLER},consdb={COL_DAYOBS_CONSDB}")

## 3. Build a canonical merged visit table

We merge Butler and consDb records, keeping one entry per unique `visitId`.
Butler is used as the primary source; consDb contributes visits absent from Butler.

The canonical table is normalised to always have columns:
`visitId`, `s_ra`, `s_dec`, `band`, `day_obs`.

In [ ]:
def normalise_visit_df(df, col_visit, col_ra, col_dec, col_band, col_dayobs, source_label):
    """
    Select and rename the relevant columns from a visit DataFrame into a
    canonical schema: visitId, s_ra, s_dec, band, day_obs.

    Parameters
    ----------
    df           : pd.DataFrame  Raw visit table
    col_visit    : str           Column holding the visit identifier
    col_ra       : str or None   Column holding pointing RA (degrees, ICRS)
    col_dec      : str or None   Column holding pointing Dec (degrees, ICRS)
    col_band     : str or None   Column holding the band / filter name
    col_dayobs   : str or None   Column holding the observation date (YYYYMMDD int)
    source_label : str           Label for printout ('butler' or 'consdb')

    Returns
    -------
    df_norm : pd.DataFrame with canonical columns
    """
    rename_map = {col_visit: "visitId"}
    keep_cols = [col_visit]

    for src, dst in [(col_ra, "s_ra"), (col_dec, "s_dec"), (col_band, "band"), (col_dayobs, "day_obs")]:
        if src is not None and src in df.columns:
            keep_cols.append(src)
            rename_map[src] = dst

    df_norm = df[keep_cols].copy().rename(columns=rename_map)
    df_norm["visitId"] = df_norm["visitId"].astype(int)

    # Remove engineering / pinhole filters if band column exists
    if "band" in df_norm.columns:
        bad_filters = {"other", "none", "other:pinhole", "pinhole"}
        df_norm = df_norm[~df_norm["band"].isin(bad_filters)]

    # Drop rows with missing pointing coordinates
    subset_drop = [c for c in ["s_ra", "s_dec"] if c in df_norm.columns]
    if subset_drop:
        df_norm = df_norm.dropna(subset=subset_drop)

    df_norm = df_norm.reset_index(drop=True)
    print(f"  [{source_label}] normalised shape: {df_norm.shape}")
    return df_norm


print("Normalising Butler visits ...")
df_b = normalise_visit_df(
    df_butler, COL_VISIT_BUTLER, COL_RA_BUTLER, COL_DEC_BUTLER, COL_BAND_BUTLER, COL_DAYOBS_BUTLER, "butler"
)

print("Normalising consDb visits ...")
df_c = normalise_visit_df(
    df_consdb, COL_VISIT_CONSDB, COL_RA_CONSDB, COL_DEC_CONSDB, COL_BAND_CONSDB, COL_DAYOBS_CONSDB, "consdb"
)

# Merge: Butler first, consDb fills in missing visits
butler_visit_ids = set(df_b["visitId"].values)
df_c_extra = df_c[~df_c["visitId"].isin(butler_visit_ids)].copy()

df_visits = pd.concat([df_b, df_c_extra], ignore_index=True)
df_visits = df_visits.drop_duplicates(subset=["visitId"]).reset_index(drop=True)

print(f"\nCanonical merged visit table: {len(df_visits):,} unique visits")
print(f"  Columns: {df_visits.columns.tolist()}")
df_visits.head(3)

### 3.1 Parse observation dates and extract year-month

In [ ]:
if "day_obs" in df_visits.columns:
    # day_obs is an integer YYYYMMDD (e.g. 20250415)
    df_visits["day_obs_str"] = df_visits["day_obs"].astype(str)
    df_visits["obs_date"] = pd.to_datetime(df_visits["day_obs_str"], format="%Y%m%d", errors="coerce")
    df_visits["year_month"] = df_visits["obs_date"].dt.to_period("M")  # e.g. 2025-04
    df_visits = df_visits.dropna(subset=["obs_date"]).reset_index(drop=True)

    months = sorted(df_visits["year_month"].dropna().unique())

    print(f"Date range : {df_visits['obs_date'].min().date()}  ->  {df_visits['obs_date'].max().date()}")
    print(f"Months found ({len(months)}) : {[str(m) for m in months]}")
else:
    print("WARNING: no day_obs column found — monthly plots will be skipped.")
    months = []

print(
    f"Bands present  : {sorted(df_visits['band'].dropna().unique()) if 'band' in df_visits.columns else 'N/A'}"
)
print(f"Total visits   : {len(df_visits):,}")

## 4. Helper functions for HEALPix maps

In [ ]:
def visits_to_healpix_map(ra_deg, dec_deg, nside=NSIDE):
    """
    Build a HEALPix visit-count map from pointing coordinates.

    Parameters
    ----------
    ra_deg, dec_deg : array-like  Pointing coordinates in degrees (ICRS)
    nside           : int         HEALPix NSIDE resolution parameter

    Returns
    -------
    hpmap : np.ndarray, shape (hp.nside2npix(nside),)
        Visit count per pixel; unobserved pixels are set to hp.UNSEEN.
    """
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    theta = np.radians(90.0 - dec)  # colatitude
    phi = np.radians(ra)
    ipix = hp.ang2pix(nside, theta, phi)
    hpmap = np.zeros(hp.nside2npix(nside), dtype=float)
    np.add.at(hpmap, ipix, 1)
    hpmap[hpmap == 0] = hp.UNSEEN
    return hpmap


def galactic_plane_icrs():
    """
    Return (ra_deg, dec_deg) arrays tracing the Galactic plane in ICRS,
    sorted by RA to avoid spurious wrap-around lines at 0/360 deg.
    """
    gal_l = np.linspace(0.0, 360.0, 1440)
    gal_b = np.zeros(1440)
    gp = SkyCoord(l=gal_l * u.deg, b=gal_b * u.deg, frame="galactic").icrs
    idx = np.argsort(gp.ra.deg)
    return gp.ra.deg[idx], gp.dec.deg[idx]


# Pre-compute the Galactic plane trace once
GP_RA_DEG, GP_DEC_DEG = galactic_plane_icrs()

print("HEALPix helper functions defined.")

## 5. Global HEALPix map — all visits, all bands, log scale

This map accumulates **all visits** over the full survey period covered by the parquet files.
The color scale is logarithmic (base-10 of visit count per pixel).
The Galactic plane (grey dots) and DDF `+` markers are overlaid.

In [ ]:
# Build the global all-bands HEALPix map
df_global = df_visits.dropna(subset=["s_ra", "s_dec"])

hpmap_global = visits_to_healpix_map(df_global["s_ra"].values, df_global["s_dec"].values, nside=NSIDE)

valid_pix = hpmap_global[hpmap_global != hp.UNSEEN]
total_visits = int(valid_pix.sum())
n_pix_obs = len(valid_pix)
sky_area_deg2 = n_pix_obs * hp.nside2pixarea(NSIDE, degrees=True)

print(f"Global HEALPix map statistics (NSIDE={NSIDE}):")
print(f"  Total visits accumulated : {total_visits:,}")
print(f"  Observed pixels          : {n_pix_obs:,}  ({sky_area_deg2:.0f} deg^2)")
print(f"  Max visits per pixel     : {int(valid_pix.max())}")
print(f"  Median visits per pixel  : {np.median(valid_pix):.1f}")

In [ ]:
# ── Plot the global map in log scale ────────────────────────────────────────

date_min = df_visits["obs_date"].min().strftime("%Y-%m-%d") if "obs_date" in df_visits.columns else "?"
date_max = df_visits["obs_date"].max().strftime("%Y-%m-%d") if "obs_date" in df_visits.columns else "?"

fig = plt.figure(figsize=(14, 7))

hp.mollview(
    hpmap_global,
    title=(
        f"{INSTRUMENT}  —  All visits, all bands  —  Log visit count  (NSIDE={NSIDE})\n"
        f"[{date_min}  to  {date_max}]   {total_visits:,} visits total"
    ),
    cmap="YlOrRd",
    norm="log",
    min=1,
    max=valid_pix.max() if len(valid_pix) > 0 else 10,
    unit="log scale (N visits)",
    bgcolor="white",
    badcolor="#eeeeee",
    flip="astro",
    coord="C",
)
hp.graticule(dpar=30, dmer=60, alpha=0.2, lw=0.4)

# Galactic plane — thin grey trace
hp.projplot(GP_RA_DEG, GP_DEC_DEG, ",", ms=0.8, color="dimgrey", alpha=0.5, lonlat=True, coord="C")

# DDF '+' markers in navy
hp.projscatter(
    DDF_RA_DEG, DDF_DEC_DEG, marker="+", s=120, c="navy", lw=1.2, zorder=10, lonlat=True, coord="C"
)

# Add DDF text labels
for name, ra, dec in zip(DDF_NAMES, DDF_RA_DEG, DDF_DEC_DEG):
    hp.projtext(ra, dec + 3.5, name, fontsize=7, color="navy", ha="center", lonlat=True, coord="C")

plt.tight_layout()
plt.show()

## 6. Monthly HEALPix maps — all-bands combined, one subplot per month

One subplot per calendar month shows the **all-band combined visit-count map** in log scale.
The grid adapts automatically to the number of months (up to 4 columns).
Reading left-to-right / top-to-bottom reveals how the Rubin survey footprint
grows over time.

In [ ]:
# Pre-compute monthly all-bands HEALPix maps
monthly_hpmaps_allbands = {}
monthly_n_visits = {}

for month in months:
    month_str = str(month)
    df_month = df_visits[df_visits["year_month"] == month].dropna(subset=["s_ra", "s_dec"])
    n_vis = len(df_month)
    monthly_n_visits[month_str] = n_vis

    if n_vis > 0:
        monthly_hpmaps_allbands[month_str] = visits_to_healpix_map(
            df_month["s_ra"].values, df_month["s_dec"].values, nside=NSIDE
        )
    else:
        monthly_hpmaps_allbands[month_str] = np.full(hp.nside2npix(NSIDE), hp.UNSEEN)

print(f"Pre-computed HEALPix maps for {len(months)} months.")
for month_str, n_vis in monthly_n_visits.items():
    print(f"  {month_str} : {n_vis:,} visits")

In [ ]:
# ── Monthly overview grid — all bands combined, log scale ───────────────────

n_months = len(months)

if n_months == 0:
    print("No months available — check that day_obs column is present.")
else:
    # Layout: up to 4 columns, rows computed automatically
    ncols = min(4, n_months)
    nrows = int(np.ceil(n_months / ncols))

    fig = plt.figure(figsize=(ncols * 5.5, nrows * 3.4))
    fig.suptitle(
        f"{INSTRUMENT}  —  All-bands combined  —  Monthly evolution  (NSIDE={NSIDE}, log scale)",
        fontsize=14,
        fontweight="bold",
        y=1.01,
    )

    for m_idx, month in enumerate(months):
        month_str = str(month)
        hpmap_month = monthly_hpmaps_allbands[month_str]
        n_vis_month = monthly_n_visits[month_str]
        sp = m_idx + 1

        valid_month = hpmap_month[hpmap_month != hp.UNSEEN]
        vmax_month = valid_month.max() if len(valid_month) > 0 else 10

        hp.mollview(
            hpmap_month,
            title=f"{month_str}\n({n_vis_month:,} visits)",
            cmap="YlOrRd",
            norm="log",
            min=1,
            max=vmax_month,
            unit="",
            bgcolor="white",
            badcolor="#eeeeee",
            flip="astro",
            coord="C",
            sub=(nrows, ncols, sp),
            margins=(0.005, 0.005, 0.005, 0.005),
        )
        hp.graticule(dpar=30, dmer=60, alpha=0.18, lw=0.3)
        hp.projplot(GP_RA_DEG, GP_DEC_DEG, ",", ms=0.7, color="dimgrey", alpha=0.45, lonlat=True, coord="C")
        hp.projscatter(
            DDF_RA_DEG, DDF_DEC_DEG, marker="+", s=80, c="navy", lw=0.8, zorder=10, lonlat=True, coord="C"
        )

    plt.tight_layout()
    plt.show()

## 7. Monthly HEALPix maps — 2×3 band panel per month (u, g, r / i, z, y)

For each calendar month, produce a 2×3 figure with one subplot per LSST band:

```
top row    :  u (Purples)  |  g (Greens)  |  r (Reds)
bottom row :  i (Oranges)  |  z (Blues)   |  y (YlOrBr)
```

Each band uses a colormap that matches its physical wavelength.  
Galactic plane (grey dots) and DDF `+` markers are overlaid.  
Log scale is applied whenever a band has observed pixels.

In [ ]:
def plot_monthly_6bands_panel(month_str, hpmaps_dict, nside=NSIDE):
    """
    Produce a 2x3 figure (u, g, r / i, z, y) for a single calendar month.

    Parameters
    ----------
    month_str   : str   e.g. '2025-06'
    hpmaps_dict : dict  {band: hpmap_array}  (missing keys -> empty map)
    nside       : int   HEALPix NSIDE parameter
    """
    empty_map = np.full(hp.nside2npix(nside), hp.UNSEEN)

    # Build per-band subtitle strings (visit count)
    subtitles = {}
    for band in BANDS:
        hpmap = hpmaps_dict.get(band, empty_map)
        valid = hpmap[hpmap != hp.UNSEEN]
        if len(valid) == 0:
            subtitles[band] = f"Band {band}  —  no data"
        else:
            subtitles[band] = f"Band {band}   ({int(valid.sum()):,} visits)"

    fig = plt.figure(figsize=(18, 8))
    fig.suptitle(
        f"{INSTRUMENT}  —  {month_str}  —  HEALPix visit counts  (NSIDE={nside}, log scale)",
        fontsize=13,
        fontweight="bold",
        y=1.01,
    )

    for row_idx, row_bands in enumerate(BAND_ROWS):
        for col_idx, band in enumerate(row_bands):
            sp = row_idx * 3 + col_idx + 1  # subplot index 1..6
            hpmap = hpmaps_dict.get(band, empty_map)
            valid = hpmap[hpmap != hp.UNSEEN]
            vmax = valid.max() if len(valid) > 0 else 10

            hp.mollview(
                hpmap,
                title=subtitles[band],
                cmap=BAND_CMAPS[band],
                norm="log",
                min=1,
                max=vmax,
                unit="log scale",
                bgcolor="white",
                badcolor="#eeeeee",
                flip="astro",
                coord="C",
                sub=(2, 3, sp),
                margins=(0.005, 0.005, 0.005, 0.005),
            )
            hp.graticule(dpar=30, dmer=60, alpha=0.18, lw=0.3)
            hp.projplot(
                GP_RA_DEG, GP_DEC_DEG, ",", ms=0.7, color="dimgrey", alpha=0.45, lonlat=True, coord="C"
            )
            hp.projscatter(
                DDF_RA_DEG,
                DDF_DEC_DEG,
                marker="+",
                s=90,
                c="black",
                lw=0.7,
                zorder=10,
                lonlat=True,
                coord="C",
            )

    plt.tight_layout()
    plt.show()


print("Band-panel plotting function defined.")

In [ ]:
# Collect per-band counts for the summary table
monthly_counts = {}  # {month_str: {band: n_visits}}

if "band" not in df_visits.columns:
    print("WARNING: no band column found — per-band monthly panels will be skipped.")
else:
    for month in months:
        month_str = str(month)
        df_month = df_visits[df_visits["year_month"] == month].dropna(subset=["s_ra", "s_dec"])
        monthly_counts[month_str] = {}
        hpmaps_bands = {}

        print("=" * 72)
        print(f"  Month : {month_str}   |   Total visits : {len(df_month):,}")
        print("=" * 72)

        for band in BANDS:
            df_mb = df_month[df_month["band"] == band]
            n_vis = len(df_mb)
            monthly_counts[month_str][band] = n_vis

            if n_vis == 0:
                print(f"    Band {band}: no visits.")
                continue

            hpmap_mb = visits_to_healpix_map(df_mb["s_ra"].values, df_mb["s_dec"].values, nside=NSIDE)
            hpmaps_bands[band] = hpmap_mb

            valid = hpmap_mb[hpmap_mb != hp.UNSEEN]
            n_pix = len(valid)
            sky_area = n_pix * hp.nside2pixarea(NSIDE, degrees=True)
            print(
                f"    Band {band}: {n_vis:5,} visits | "
                f"{n_pix:4d} pixels | "
                f"sky {sky_area:.0f} deg^2 | "
                f"max {int(valid.max())} vis/pix"
            )

        # Produce the 2x3 panel for this month
        if hpmaps_bands:
            plot_monthly_6bands_panel(month_str, hpmaps_bands, nside=NSIDE)

        print()

## 8. Monthly visit counts — summary table and bar charts

In [ ]:
if monthly_counts:
    df_monthly = pd.DataFrame(monthly_counts).T.fillna(0).astype(int)
    df_monthly.index.name = "Month"

    # Ensure all 6 bands appear even if some are missing
    for b in BANDS:
        if b not in df_monthly.columns:
            df_monthly[b] = 0
    df_monthly = df_monthly[BANDS]
    df_monthly["Total"] = df_monthly[BANDS].sum(axis=1)

    print("Monthly visit counts per band:")
    display(df_monthly)
else:
    print("No monthly counts available.")
    df_monthly = None

In [ ]:
# ── Grouped bar chart: one bar per band per month ────────────────────────────

if monthly_counts and months:
    month_labels = [str(m) for m in months]
    x = np.arange(len(month_labels))
    bar_w = 0.12
    n_bands = len(BANDS)
    offsets = np.linspace(-(n_bands - 1) / 2 * bar_w, (n_bands - 1) / 2 * bar_w, n_bands)

    fig, ax = plt.subplots(figsize=(max(10, len(months) * 1.6), 6))

    for band, offset in zip(BANDS, offsets):
        counts = [monthly_counts.get(m, {}).get(band, 0) for m in month_labels]
        ax.bar(
            x + offset,
            counts,
            width=bar_w,
            label=f"Band {band}",
            color=BAND_COLORS[band],
            edgecolor="white",
            linewidth=0.5,
            alpha=0.92,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(month_labels, rotation=30, ha="right", fontsize=11)
    ax.set_xlabel("Month", fontsize=13)
    ax.set_ylabel("Number of visits", fontsize=13)
    ax.set_title(
        f"{INSTRUMENT}  —  Monthly visit counts per LSST band",
        fontsize=14,
        fontweight="bold",
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda val, _: f"{int(val):,}"))
    ax.legend(title="Band", fontsize=10, loc="upper left", framealpha=0.85)
    ax.grid(axis="y", alpha=0.35, linestyle="--")
    ax.set_xlim(x[0] - 0.5, x[-1] + 0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Stacked bar chart ────────────────────────────────────────────────────────

if monthly_counts and months:
    fig, ax = plt.subplots(figsize=(max(10, len(months) * 1.6), 6))
    bottom = np.zeros(len(month_labels))

    for band in BANDS:
        counts = np.array([monthly_counts.get(m, {}).get(band, 0) for m in month_labels], dtype=float)
        ax.bar(
            x,
            counts,
            bottom=bottom,
            label=f"Band {band}",
            color=BAND_COLORS[band],
            edgecolor="white",
            linewidth=0.5,
            alpha=0.92,
        )
        bottom += counts

    # Annotate total on top of each stacked bar
    for i, tot in enumerate(bottom):
        if tot > 0:
            ax.text(
                x[i],
                tot * 1.012,
                f"{int(tot):,}",
                ha="center",
                va="bottom",
                fontsize=9,
                fontweight="bold",
            )

    ax.set_xticks(x)
    ax.set_xticklabels(month_labels, rotation=30, ha="right", fontsize=11)
    ax.set_xlabel("Month", fontsize=13)
    ax.set_ylabel("Number of visits (stacked)", fontsize=13)
    ax.set_title(
        f"{INSTRUMENT}  —  Monthly visit counts per LSST band (stacked)",
        fontsize=14,
        fontweight="bold",
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda val, _: f"{int(val):,}"))
    ax.legend(title="Band", fontsize=10, loc="upper left", framealpha=0.85)
    ax.grid(axis="y", alpha=0.35, linestyle="--")
    ax.set_xlim(x[0] - 0.5, x[-1] + 0.5)
    plt.tight_layout()
    plt.show()

## 9. Summary

| Item | Value |
|------|-------|
| Butler visits loaded | *(fill from output above)* |
| consDb visits loaded | *(fill from output above)* |
| Canonical merged visits | *(fill from output above)* |
| Survey date range | *(fill from output above)* |
| Number of months | *(fill from output above)* |
| HEALPix NSIDE | `64` |
| Total visits in global map | *(fill from output above)* |
| Sky area observed (deg²) | *(fill from output above)* |